In [0]:
# Databricks notebook source

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import to_date, date_format


def create_gold_fact_sales(sales_order_detail_df, sales_order_header_df):

    sales_order_detail_df = sales_order_detail_df.select("SalesOrderID", "SalesOrderDetailID", "ProductID", "OrderQty", "UnitPrice", "UnitPriceDiscount", "LineTotal")
    sales_order_header_df = sales_order_header_df.select("SalesOrderID",  "OrderDate", "CustomerID", "SalesPersonID", "TerritoryID", "TaxAmt", "Freight", "TotalDue")
    fact_sales_tgt_df = sales_order_detail_df.join(sales_order_header_df, sales_order_detail_df.SalesOrderID == sales_order_header_df.SalesOrderID, "inner").drop(sales_order_header_df.SalesOrderID).dropDuplicates(["SalesOrderDetailID"]).withColumn("processed_timestamp", F.current_timestamp())
    
    fact_sales_tgt_df = fact_sales_tgt_df.select("SalesOrderID", "SalesOrderDetailID", "ProductID","CustomerID", "SalesPersonID", "TerritoryID","OrderQty", "UnitPrice", "UnitPriceDiscount", "LineTotal","OrderDate","TaxAmt", "Freight", "TotalDue")
                                 
    return fact_sales_tgt_df



if __name__ == "__main__":

    sales_order_detail_tbl = dbutils.widgets.get("sales_order_detail")
    sales_order_header_tbl = dbutils.widgets.get("sales_order_header")
    sales_order_detail_df = df = spark.read.table(sales_order_detail_tbl)
    sales_order_header_df = df = spark.read.table(sales_order_header_tbl)

    fact_sales_tgt_df = create_gold_fact_sales(sales_order_detail_df, sales_order_header_df)
    fact_sales_gold_tbl = dbutils.widgets.get("fact_sales")
    fact_sales_tgt_df.write.mode("overwrite").format("delta").partitionBy("OrderDate").saveAsTable(fact_sales_gold_tbl)